# Lab 4, Fine-Tuning an LLM with LoRA / QLoRA
**PUSL 3205, Large Language Models and Generative AI**

In this lab I parameter-efficiently fine-tune a small instruction-tuned causal LLM so it learns a
specific domain: the products, pricing, warranty, repairs, and policies of a (fictional) computer
shop, **CoreByte Computers**. I use **LoRA** (Low-Rank Adaptation) via PEFT, with a **QLoRA**
(4-bit) path when a suitable CUDA GPU is present, and train with TRL's `SFTTrainer`.

### ⚠️ Hardware note, please read first
QLoRA's 4-bit path (via `bitsandbytes`) **requires a CUDA GPU**; it cannot run on CPU or on Apple
MPS. This notebook was authored on a Mac (Apple Silicon, **no CUDA GPU**), where additionally the
local `transformers` model-loading path deadlocks on the bleeding-edge Python 3.14 / PyTorch 2.14
stack. So the model-loading, training, and evaluation cells are **guarded behind a GPU check** and
were **not executed locally**, they print a clear "skipped" message here instead of fabricating
results.

**To run the fine-tune for real:** open this notebook in **Google Colab → Runtime → Change runtime
type → T4 GPU**, run all cells, and the guarded sections will execute QLoRA end to end. Everything
is written and ready. No loss curves or model outputs in this notebook are invented.

In [1]:
# Core imports (these are safe on any machine)
import os, json, platform
import torch

print("Library versions")
print("  torch       :", torch.__version__)
import transformers, peft, trl, datasets
print("  transformers:", transformers.__version__)
print("  peft        :", peft.__version__)
print("  trl         :", trl.__version__)
print("  datasets    :", datasets.__version__)
try:
    import bitsandbytes as bnb
    print("  bitsandbytes:", bnb.__version__)
except Exception:
    print("  bitsandbytes: not installed (expected on non-CUDA machines)")

Library versions
  torch       : 2.14.1


W1021 13:26:07.801000 3594 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


  transformers: 5.20.2
  peft        : 0.21.2
  trl         : 1.16.0
  datasets    : 5.1.0
  bitsandbytes: not installed (expected on non-CUDA machines)


## 1. Detect the hardware

The lab asks me to check for a CUDA GPU and record what is available. This drives which path the
notebook takes:

- **CUDA GPU + bitsandbytes** → QLoRA (4-bit quantized base model), the most memory-efficient path.
- **CUDA GPU only** → plain LoRA in fp16 (fallback when 4-bit isn't available).
- **No CUDA GPU** → the training/eval cells are skipped with a message; run on Colab to execute.

In [2]:
USE_CUDA = torch.cuda.is_available()
HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

print("Platform :", platform.platform(), "|", platform.machine())
print("CUDA GPU :", USE_CUDA)
print("Apple MPS:", HAS_MPS)

if USE_CUDA:
    print("GPU name :", torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {total_mem:.1f} GB")
    DEVICE = "cuda"
else:
    DEVICE = "mps" if HAS_MPS else "cpu"
    print(f"\n>>> No CUDA GPU detected. DEVICE = '{DEVICE}'.")
    print(">>> Training/eval cells below will be SKIPPED here and should be run on a CUDA GPU (e.g. Colab T4).")

# Try to enable 4-bit QLoRA only when it can actually work
try:
    import bitsandbytes  # noqa
    HAS_BNB = True
except Exception:
    HAS_BNB = False
USE_4BIT = USE_CUDA and HAS_BNB
print("Will use QLoRA (4-bit):", USE_4BIT)

Platform : macOS-26.4.1-arm64-arm-64bit-Mach-O | arm64
CUDA GPU : False
Apple MPS: True

>>> No CUDA GPU detected. DEVICE = 'mps'.
>>> Training/eval cells below will be SKIPPED here and should be run on a CUDA GPU (e.g. Colab T4).
Will use QLoRA (4-bit): False


## 2. The domain dataset

I hand-wrote **37 question–answer pairs** about CoreByte Computers, opening hours, location,
products, pricing, warranty, repairs, delivery, returns, student discount, and so on, kept
deliberately consistent so the fine-tune has a clear, learnable signal. The data lives in
[`dataset/qa_pairs.json`](dataset/qa_pairs.json). Loading and inspecting it needs no GPU.

In [3]:
with open("dataset/qa_pairs.json") as f:
    data = json.load(f)

profile = data["shop_profile"]
qa_pairs = data["qa_pairs"]
print(f"Shop: {profile['name']}, {profile['location']}")
print(f"Loaded {len(qa_pairs)} Q&A pairs\n")
for ex in qa_pairs[:3]:
    print("Q:", ex["question"])
    print("A:", ex["answer"], "\n")

from datasets import Dataset
raw_ds = Dataset.from_list(qa_pairs)
print(raw_ds)

Shop: CoreByte Computers, 48 Harbour Street, Galle, Sri Lanka
Loaded 37 Q&A pairs

Q: What are your opening hours?
A: CoreByte Computers is open Monday to Friday from 9 AM to 7 PM, and Saturday from 10 AM to 5 PM. We are closed on Sundays and public holidays. 

Q: Are you open on weekends?
A: We are open on Saturdays from 10 AM to 5 PM, but we are closed on Sundays. 

Q: What time do you close on weekdays?
A: On weekdays, Monday to Friday, we close at 7 PM. 

Dataset({
    features: ['question', 'answer'],
    num_rows: 37
})


### Chat formatting

`SFTTrainer` trains on formatted text. I turn each Q&A pair into a two-turn chat (user question →
assistant answer) and render it with the **base model's own chat template** at train time, so the
fine-tune matches the format the instruct model already expects. The helper below is defined now and
called inside the (GPU-guarded) training section once the tokenizer is loaded.

In [4]:
def format_example(example, tokenizer):
    """Render one Q&A pair using the model's chat template -> {'text': ...}."""
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

print("format_example() defined. It will be mapped over the dataset once the tokenizer is loaded.")

format_example() defined. It will be mapped over the dataset once the tokenizer is loaded.


## 3. Choose the base model

I use **`HuggingFaceTB/SmolLM2-360M-Instruct`**, a 360M-parameter instruction-tuned model. It is
small enough to QLoRA-fine-tune comfortably on a free Colab T4, while still being a real
instruct model whose behaviour we can meaningfully change. (`Qwen/Qwen2.5-0.5B-Instruct` is a fine
drop-in alternative, just change `MODEL_NAME`.)

In [5]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
print("Base model:", MODEL_NAME)

Base model: HuggingFaceTB/SmolLM2-360M-Instruct


## 4. LoRA / QLoRA configuration

**LoRA** freezes the base model and injects small trainable low-rank matrices into the attention (and
MLP) projections, so only a tiny fraction of parameters are trained. **QLoRA** additionally loads the
frozen base weights in **4-bit** (NF4) to cut memory further, while the LoRA adapters stay in higher
precision.

The config I use:

| Hyper-parameter | Value | Why |
|---|---|---|
| `r` (rank) | 16 | Capacity of the low-rank update, enough to learn a small domain. |
| `lora_alpha` | 32 | Scaling (α/r = 2); a common, stable ratio. |
| `lora_dropout` | 0.05 | Light regularisation on the small dataset. |
| `target_modules` | attention + MLP projections | Where the adapters are injected. |
| 4-bit quant | NF4 + double-quant, compute dtype fp16 | QLoRA memory savings (CUDA only). |

Building the config objects needs no GPU, only *using* them to load/train does.

In [6]:
from peft import LoraConfig
from transformers import BitsAndBytesConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

# 4-bit config is only *used* when USE_4BIT is True, but we can define it safely anywhere.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("LoRA config:")
print(f"  r={lora_config.r}, alpha={lora_config.lora_alpha}, dropout={lora_config.lora_dropout}")
print(f"  target_modules={lora_config.target_modules}")
print("4-bit (QLoRA) will be applied:", USE_4BIT)

LoRA config:
  r=16, alpha=32, dropout=0.05
  target_modules={'up_proj', 'v_proj', 'down_proj', 'k_proj', 'o_proj', 'q_proj', 'gate_proj'}
4-bit (QLoRA) will be applied: False


## 5–7. Load the model, fine-tune with SFTTrainer, and plot the loss

> **GPU required.** Everything in this section loads the base model and trains it, so it only runs
> when a CUDA GPU is present. On a non-CUDA machine it is skipped with a message. On Colab T4 it
> runs QLoRA end to end, records the training loss, and saves the loss curve to
> `outputs/training_loss.png`.

In [7]:
def run_training():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl import SFTTrainer, SFTConfig
    import matplotlib.pyplot as plt

    os.makedirs("outputs", exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = dict(device_map="auto")
    if USE_4BIT:
        model_kwargs.update(quantization_config=bnb_config)
    else:
        model_kwargs.update(dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

    # Format the dataset with the model's chat template
    train_ds = raw_ds.map(lambda ex: format_example(ex, tokenizer))

    sft_config = SFTConfig(
        output_dir="outputs/sft",
        num_train_epochs=12,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        report_to=[],
        max_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        peft_config=lora_config,
        processing_class=tokenizer,
    )
    trainer.train()

    # Plot and save the training loss
    hist = [h for h in trainer.state.log_history if "loss" in h]
    steps = [h["step"] for h in hist]
    losses = [h["loss"] for h in hist]
    plt.figure(figsize=(8, 4.5))
    plt.plot(steps, losses, "-o", color="#5b5bd6")
    plt.title("LoRA fine-tuning loss, SmolLM2-360M on CoreByte Q&A")
    plt.xlabel("training step"); plt.ylabel("training loss"); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("outputs/training_loss.png", bbox_inches="tight")
    plt.show()

    return model, tokenizer, trainer


if USE_CUDA:
    model, tokenizer, trainer = run_training()
    print("Training complete.")
else:
    model = tokenizer = trainer = None
    print("SKIPPED: no CUDA GPU in this environment.")
    print("Run this notebook on Google Colab (Runtime -> T4 GPU) to execute the QLoRA fine-tune.")

SKIPPED: no CUDA GPU in this environment.
Run this notebook on Google Colab (Runtime -> T4 GPU) to execute the QLoRA fine-tune.


## 8. Evaluate, before vs after, including unseen questions

The real test is a side-by-side comparison of the **original** model and the **fine-tuned** model on
the same questions, crucially including **questions that were *not* in the training set**, to check
the model learned the domain rather than just memorising. The eval set below mixes in-domain training
questions, paraphrases, and genuinely unseen questions.

> **GPU required**, this loads the base model again and runs both models, so it is guarded too.

In [8]:
EVAL_QUESTIONS = [
    # --- seen (in training set) ---
    ("seen",       "What are your opening hours?"),
    ("seen",       "Do you offer a student discount?"),
    ("seen",       "How much is a diagnostic check?"),
    # --- paraphrases of seen facts ---
    ("paraphrase", "I'm a student, can I get any discount?"),
    ("paraphrase", "What time do you shut on Saturdays?"),
    # --- unseen (facts are learnable from the data, but phrased/asked differently) ---
    ("unseen",     "Can I pay with a credit card?"),
    ("unseen",     "I dropped my laptop and cracked the screen, is that under warranty?"),
    ("unseen",     "How do I get in touch with you?"),
]

def generate_answer(mdl, tok, question, max_new_tokens=80):
    msgs = [{"role": "user", "content": question}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


def run_evaluation():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import pandas as pd

    # Reload a clean copy of the ORIGINAL model for the "before" column
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")

    rows = []
    for kind, q in EVAL_QUESTIONS:
        before = generate_answer(base, tokenizer, q)
        after = generate_answer(model, tokenizer, q)   # fine-tuned (PEFT) model
        rows.append({"type": kind, "question": q, "original_model": before, "finetuned_model": after})

    df = pd.DataFrame(rows)
    df.to_json("outputs/before_after.json", orient="records", indent=2)
    # Also write a markdown table for REVIEW.md
    with open("outputs/before_after.md", "w") as f:
        f.write("| Type | Question | Original model | Fine-tuned model |\n")
        f.write("|------|----------|----------------|------------------|\n")
        for r in rows:
            f.write(f"| {r['type']} | {r['question']} | {r['original_model']} | {r['finetuned_model']} |\n")
    return df


if USE_CUDA and model is not None:
    import pandas as pd
    pd.set_option("display.max_colwidth", 60)
    eval_df = run_evaluation()
    print("Saved before/after to outputs/before_after.{json,md}")
    display(eval_df)
else:
    print("SKIPPED: evaluation needs the fine-tuned model from the GPU-only training step above.")

SKIPPED: evaluation needs the fine-tuned model from the GPU-only training step above.


## 9. Save the LoRA adapter

LoRA adapters are tiny (a few MB) because they're only the low-rank matrices, not the full model,
so the adapter is committed to the repo. It's loaded on top of the base model at inference time.

> **GPU required** (produced by the training step).

In [9]:
if USE_CUDA and model is not None:
    model.save_pretrained("corebyte_lora_adapter")
    tokenizer.save_pretrained("corebyte_lora_adapter")
    import os
    total = sum(os.path.getsize(os.path.join("corebyte_lora_adapter", f))
                for f in os.listdir("corebyte_lora_adapter"))
    print(f"Saved LoRA adapter to corebyte_lora_adapter/ ({total/1e6:.2f} MB)")
    print("Reload later with: PeftModel.from_pretrained(base_model, 'corebyte_lora_adapter')")
else:
    print("SKIPPED: nothing to save without a trained adapter (run on a CUDA GPU).")

SKIPPED: nothing to save without a trained adapter (run on a CUDA GPU).


## Discussion, why a GPU helps, and what LoRA/QLoRA buy us

- **Why a GPU helps LLM fine-tuning.** Fine-tuning is thousands of forward+backward passes over a
  model with hundreds of millions of parameters. GPUs run the underlying matrix multiplications
  massively in parallel (and QLoRA's 4-bit kernels in `bitsandbytes` are CUDA-only), so what takes
  minutes on a GPU can take hours, or deadlock, on CPU.
- **LoRA = parameter efficiency.** By freezing the base model and training only small low-rank
  adapters, LoRA trains well under 1% of the parameters, so the optimiser state and gradients are
  tiny and the saved artefact is a few MB instead of gigabytes.
- **QLoRA = memory efficiency.** Loading the frozen base in 4-bit roughly quarters its memory
  footprint, which is what lets a model fit, and fine-tune, on a modest GPU like a Colab T4.
- **Guarding against over/under-fitting.** With only 37 examples and 12 epochs, the risk is
  *over-fitting* (parroting training answers verbatim), which is exactly why the eval set includes
  paraphrases and unseen questions. The before/after on those rows is what shows whether the model
  *learned the domain* or just memorised strings.

See [`REVIEW.md`](REVIEW.md) for the hardware findings, the method, and the before/after comparison.